# refactor

> Move a top-level definition to another file and repoint every import that named it, on top of the index that already knows where the symbol lives.

In [ ]:
#| default_exp refactor

In [ ]:
#| hide
from fastcore.test import *
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import ast, builtins, re, textwrap
from dataclasses import dataclass
from functools import partial
from itertools import accumulate
from fastcore.all import Path, first
from kosha.core import imp_root

In [ ]:
#| hide
from tempfile import TemporaryDirectory

def _tree(d, files):
    "Write a small multi-file project under `d` and hand back its root."
    for k, v in files.items():
        p = Path(d)/k; p.parent.mkdir(parents=True, exist_ok=True); p.write_text(textwrap.dedent(v).lstrip('\n'))
    return Path(d)

## Edits

Every plan is a list of whole files: the text before, the text after, and how many changes it took. Nothing is written, so a caller can show a diff, apply it, and undo it by writing `before` back. A `before` of `None` means the file did not exist, which is what makes undo delete it again.

In [ ]:
#| export
_BUILTINS = frozenset(dir(builtins))

@dataclass
class FileEdit:
    "One file's whole text before and after, so a plan can be shown, applied, and undone."
    path: str
    before: str
    after: str
    edits: int
    def dict(self): return dict(path=self.path, before=self.before, after=self.after, edits=self.edits)

def _read(path):
    "A file's text, or None when it does not exist."
    p = Path(path)
    return p.read_text(encoding='utf-8', errors='replace') if p.is_file() else None

In [ ]:
FileEdit('a.py', 'x = 1\n', 'y = 1\n', 1).dict()

{'path': 'a.py', 'before': 'x = 1\n', 'after': 'y = 1\n', 'edits': 1}

In [ ]:
#| hide
with TemporaryDirectory() as d:
    test_eq(_read(Path(d)/'missing.py'), None)
    test_eq(_read(_tree(d, {'a.py': 'x = 1\n'})/'a.py'), 'x = 1\n')

## Text replacement

`replace_plan` takes paths and a `read`, not a directory, so it works against a working tree, a git revision, or an editor's unsaved buffers. Reading with `errors='replace'` turns a binary file into replacement characters, and a file carrying one is skipped rather than corrupted.

In [ ]:
#| export
def _pattern(query, regex=False, case=False, word=False):
    if not query: raise ValueError('find text is empty')
    text = query if regex else re.escape(query)
    if word: text = r'\b' + text + r'\b'
    try: return re.compile(text, 0 if case else re.I)
    except re.error as e: raise ValueError(f'invalid regular expression: {e}') from e

def replace_plan(paths, query, replacement, regex=False, case=False, word=False, read=_read):
    "Changed files for one literal or regular-expression replacement, and the paths left unread."
    pat = _pattern(str(query), bool(regex), bool(case), bool(word))
    repl = str(replacement) if regex else str(replacement).replace('\\', '\\\\')
    rows, skipped = [], []
    for path in paths:
        if Path(path).suffix.lower() == '.ipynb': skipped.append(str(path)); continue
        try: before = read(path)
        except Exception: before = None
        if before is None or '�' in before: skipped.append(str(path)); continue
        try: after, n = pat.subn(repl, before)
        except re.error as e: raise ValueError(f'invalid replacement: {e}') from e
        if n: rows.append(FileEdit(str(path), before, after, n))
    return rows, skipped

In [ ]:
with TemporaryDirectory() as d:
    root = _tree(d, {'a.py': 'kosha = 1\nprint(koshas)\n', 'b.py': 'x = 2\n', 'nb.ipynb': '{}'})
    rows, skipped = replace_plan(sorted(root.iterdir()), 'kosha', 'kosha2', word=True)
    print([Path(r.path).name for r in rows], [Path(p).name for p in skipped])
    print(rows[0].after)

['a.py'] ['nb.ipynb']
kosha2 = 1
print(koshas)



In [ ]:
#| hide
with TemporaryDirectory() as d:
    root = _tree(d, {'a.py': 'kosha = 1\nprint(koshas)\n'})
    ps = [root/'a.py']
    test_eq(replace_plan(ps, 'kosha', 'k')[0][0].edits, 2)                    # no word boundary: both
    test_eq(replace_plan(ps, 'KOSHA', 'k')[0][0].edits, 2)                    # case-insensitive by default
    test_eq(replace_plan(ps, 'KOSHA', 'k', case=True)[0], [])                 # case=True finds nothing
    test_eq(replace_plan(ps, r'kosha\d?', 'k', regex=True)[0][0].edits, 2)
    test_eq(replace_plan(ps, 'kosha', r'a\b')[0][0].after, 'a\\b = 1\nprint(a\\bs)\n')   # literal, not an escape
    test_eq(replace_plan([root/'gone.py'], 'kosha', 'k'), ([], [str(root/'gone.py')]))
    (root/'bin.dat').write_bytes(b'kosha\x00\xff\xfe')
    test_eq(replace_plan([root/'bin.dat'], 'kosha', 'k')[1], [str(root/'bin.dat')])      # undecodable: skipped
test_fail(lambda: replace_plan([], '', 'x'), contains='find text is empty')
test_fail(lambda: replace_plan([], '(', 'x', regex=True), contains='invalid regular expression')

## Symbols and their spans

`ast` reports positions as line and column, and `col_offset` counts utf-8 bytes rather than characters, so every offset here goes through `_at`.

In [ ]:
#| export
def _starts(source):
    "Character offset at which each line begins, plus the end of the text."
    return list(accumulate(map(len, source.splitlines(True)), initial=0))

def _at(source, starts, node):
    "A node's (start, end) character offsets. `col_offset` counts utf-8 bytes, not characters."
    def one(lineno, col):
        begin = starts[lineno - 1]
        return begin + len(source[begin:starts[lineno]].encode()[:col].decode('utf-8', 'ignore'))
    return one(node.lineno, node.col_offset), one(node.end_lineno, node.end_col_offset)

def _parse(path, source):
    try: return ast.parse(source)
    except SyntaxError as e: raise ValueError(f'{Path(path).name} does not parse: {e.msg} (line {e.lineno})') from e

In [ ]:
#| hide
_s = 'x = "éé"\ny = 1\n'
test_eq(_s[slice(*_at(_s, _starts(_s), ast.parse(_s).body[1]))], 'y = 1')
test_fail(lambda: _parse('bad.py', 'def ('), contains='bad.py does not parse')

A definition owns more than its own lines: the comment block above it and its decorators come with it, and so do the blank lines below it, so the file it leaves behind does not grow a gap where it was.

In [ ]:
#| export
def _span(source, starts, node):
    "What a definition owns: the comments above it, its decorators, and the blank lines below."
    lines = source.splitlines()
    top = min([d.lineno for d in getattr(node, 'decorator_list', [])] + [node.lineno])
    while top > 1 and lines[top - 2].lstrip().startswith('#'): top -= 1
    end = node.end_lineno
    while end < len(lines) and not lines[end].strip(): end += 1
    return starts[top - 1], starts[end]

def top_symbols(source):
    "Top-level functions and classes, each with the character span it owns."
    tree, starts = ast.parse(source), _starts(source)
    kind = lambda n: 'class' if isinstance(n, ast.ClassDef) else 'function'
    return [dict(name=n.name, kind=kind(n), line=n.lineno, **dict(zip(('start', 'end'), _span(source, starts, n))))
            for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))]

In [ ]:
SRC = '''import os

# the separator, repeated
@cache
def sep(n): return os.sep * n


class Thing:
    def go(self): return sep(1)
'''
top_symbols(SRC)

[{'name': 'sep', 'kind': 'function', 'line': 5, 'start': 11, 'end': 76},
 {'name': 'Thing', 'kind': 'class', 'line': 8, 'start': 76, 'end': 121}]

In [ ]:
sym = top_symbols(SRC)[0]
print(SRC[sym['start']:sym['end']])

# the separator, repeated
@cache
def sep(n): return os.sep * n





In [ ]:
#| hide
sep, thing = top_symbols(SRC)
test_eq([s['name'] for s in (sep, thing)], ['sep', 'Thing'])
test_eq([s['kind'] for s in (sep, thing)], ['function', 'class'])
test_eq(SRC[sep['start']:sep['end']].startswith('# the separator'), True)   # comment and decorator come too
test_eq(SRC[sep['end']:thing['start']], '')                                # the blank lines below go with it
test_eq(top_symbols('x = 1\n'), [])                                        # assignments are not symbols

## Naming the module a file is

`module_of` reuses `kosha.core.imp_root`, which walks up while a directory still holds an `__init__.py`, so a file's dotted name is decided the same way here as it is when the index records `mod_name`. A package's `__init__.py` is the package itself, not a `pkg.__init__` submodule.

In [ ]:
#| export
def module_of(path):
    "Dotted module name for a file, from the top of its `__init__.py` chain."
    p = Path(path)
    parts = p.relative_to(imp_root(p)).with_suffix('').parts
    return '.'.join(parts[:-1] if p.stem == '__init__' else parts)

In [ ]:
with TemporaryDirectory() as d:
    root = _tree(d, {'pkg/__init__.py': '', 'pkg/sub/__init__.py': '', 'pkg/sub/deep.py': '', 'loose.py': ''})
    print([module_of(root/p) for p in ('pkg/sub/deep.py', 'pkg/__init__.py', 'loose.py')])

['pkg.sub.deep', 'pkg', 'loose']


## Free names

The whole of moving code safely is knowing which names it reads but does not bind: those have to be reachable wherever it lands. `exclude` is normally the builtins, but a module that defines its own `filter` or `id` has to hand that set the name back, or the check would call a shadowed definition "already available".

In [ ]:
#| export
def _bound(tree):
    "Every name a module binds at its top level, mapped to the statement that binds it."
    out = {}
    for n in tree.body:
        if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): out[n.name] = n
        elif isinstance(n, (ast.Import, ast.ImportFrom)):
            for a in n.names: out[(a.asname or a.name).split('.')[0]] = n
        elif isinstance(n, ast.Assign):
            for t in n.targets:
                for x in ast.walk(t):
                    if isinstance(x, ast.Name): out[x.id] = n
        elif isinstance(n, ast.AnnAssign) and isinstance(n.target, ast.Name): out[n.target.id] = n
    return out

def _free(nodes, exclude=_BUILTINS):
    "Names the nodes read without binding, which is what has to reach them wherever they go."
    load, store = set(), set()
    for node in nodes:
        for n in ast.walk(node):
            if isinstance(n, ast.Name): (load if isinstance(n.ctx, ast.Load) else store).add(n.id)
            elif isinstance(n, ast.arg): store.add(n.arg)
            elif isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)): store.add(n.name)
            elif isinstance(n, ast.alias): store.add((n.asname or n.name).split('.')[0])
            elif isinstance(n, ast.ExceptHandler) and n.name: store.add(n.name)
            elif isinstance(n, ast.Global): store.update(n.names)
    return load - store - set(exclude)

In [ ]:
_body = ast.parse('def f(n):\n    total = BASE\n    return len(total) + n + os.sep\n').body
print(sorted(_free(_body)), '|', sorted(_free(_body, exclude=()) & {'len'}))

['BASE', 'os'] | ['len']


In [ ]:
#| hide
test_eq(_free(_body), {'BASE', 'os'})                                  # `len` is a builtin, `n` and `total` are bound
test_eq(sorted(_bound(ast.parse('import os\nX = 1\ndef f(): pass\nclass C: pass\n'))), ['C', 'X', 'f', 'os'])
test_eq(_free(ast.parse('try: pass\nexcept E as e: print(e)\n').body), {'E'})

## Extract

`extract` lifts a character selection into a function, a method, a variable, or a module-level constant. It refuses a selection it cannot lift rather than guessing: a partial statement, or one that returns or yields, is an error and not a mangled file.

In [ ]:
#| export
def _name(name):
    if not str(name).isidentifier(): raise ValueError('name must be a valid Python identifier')
    return str(name)

def _line(source, pos): return source.count('\n', 0, pos), source.rfind('\n', 0, pos) + 1

def _indent(source, pos):
    _, start = _line(source, pos)
    return re.match(r'[ \t]*', source[start:]).group()

def _expr(text):
    try: return ast.parse(text, mode='eval').body
    except SyntaxError as e: raise ValueError('select one complete Python expression') from e

def _writes(nodes):
    "Every name the selection binds."
    return {n.id for node in nodes for n in ast.walk(node)
            if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Store)}

def _reads(text):
    """Every name the text loads, so a name still wanted afterwards can be told from a dead one.

    The tail of a file is the middle of a block and rarely parses. The identifier fallback errs
    towards handing a value back, which is untidy rather than broken."""
    for candidate in (text, textwrap.dedent(text)):
        try: tree = ast.parse(candidate)
        except SyntaxError: continue
        return {n.id for n in ast.walk(tree) if isinstance(n, ast.Name) and isinstance(n.ctx, ast.Load)}
    return set(re.findall(r'\b[A-Za-z_][A-Za-z0-9_]*\b', text))

In [ ]:
#| export
#: Where a statement reads before it binds. `ast.walk` is breadth-first, so it would see the target
#: of `total = total + 1` before the value and call the name already bound.
_EVAL_ORDER = {ast.Assign: ('value', 'targets'), ast.AugAssign: ('value', 'target'),
               ast.AnnAssign: ('value', 'target'), ast.For: ('iter', 'target'),
               ast.AsyncFor: ('iter', 'target'), ast.comprehension: ('iter', 'target'),
               ast.withitem: ('context_expr', 'optional_vars')}

def _children(value):
    if isinstance(value, list): return [x for x in value if isinstance(x, ast.AST)]
    return [value] if isinstance(value, ast.AST) else []

def _reads_before_writes(nodes):
    """Names the selection reads before it binds them: those have to arrive as parameters.

    A name both read and bound, as in `total = total + n`, is a parameter as well as a result.
    Subtracting the bound names from the read ones would miss it."""
    bound, params = set(), []
    def visit(node):
        if isinstance(node, ast.Name):
            if isinstance(node.ctx, ast.Load):
                if node.id not in bound and node.id not in params: params.append(node.id)
            else: bound.add(node.id)
            return
        if (order := _EVAL_ORDER.get(type(node))) is not None:
            for field in order:
                for child in _children(getattr(node, field, None)): visit(child)
            for name, value in ast.iter_fields(node):
                if name in order: continue
                for child in _children(value): visit(child)
            return
        for child in ast.iter_child_nodes(node): visit(child)
    for node in nodes: visit(node)
    return sorted(params)

In [ ]:
#| hide
test_eq(_reads_before_writes(ast.parse('total = total + n\n').body), ['n', 'total'])   # read and bound: both
test_eq(_reads_before_writes(ast.parse('for i in xs: fn(i)\n').body), ['fn', 'xs'])
test_eq(_writes(ast.parse('a, b = 1, 2\n').body), {'a', 'b'})
test_eq(_reads('    return b + 1\n'), {'b'})                                            # dedents to parse
test_eq(_reads('    x = y\n    if (\n'), {'x', 'y', 'if'})   # unparseable: every identifier, keywords and all

In [ ]:
#| export
def _enclosing_method(source, pos):
    "The method `pos` sits in and the indent its siblings use, or a reason there is none."
    tree, starts = ast.parse(source), _starts(source)
    for cls in [n for n in ast.walk(tree) if isinstance(n, ast.ClassDef)]:
        for fn in [n for n in cls.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef))]:
            begin, stop = _at(source, starts, fn)
            if begin <= pos <= stop: return fn, ' ' * fn.col_offset
    raise ValueError("extract a method from inside one of a class's own methods")

def _offset_after(source, node):
    "Where a sibling of `node` goes: the end of the statement it follows."
    starts = _starts(source)
    for fn in [n for n in ast.walk(ast.parse(source))
               if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)) and n.name == node.name]:
        return _at(source, starts, fn)[1]
    return len(source)

In [ ]:
#| export
def extract(path, source, start, end, kind, name):
    "One safe single-file extraction; unsupported selections fail rather than guess."
    start, end, name = int(start), int(end), _name(name)
    if not 0 <= start < end <= len(source): raise ValueError('select code before extracting')
    text, indent = source[start:end], _indent(source, start)
    if kind in {'variable', 'constant'}:
        node = _expr(text)
        if kind == 'constant':
            try: ast.literal_eval(node)
            except Exception as e: raise ValueError('constants must be literal values') from e
            name = name.upper()
            if not re.fullmatch(r'[A-Z_][A-Z0-9_]*', name): raise ValueError('constant names use UPPER_CASE')
            if indent: raise ValueError('extract constants from module scope')
            tree = ast.parse(source); at = 0
            if tree.body and isinstance(tree.body[0], ast.Expr) and isinstance(tree.body[0].value, ast.Constant): at = tree.body[0].end_lineno
            while at < len(source.splitlines()) and source.splitlines()[at].startswith(('import ', 'from ')): at += 1
            point = sum(len(x) + 1 for x in source.splitlines()[:at])
            after = source[:point] + f'{name} = {text}\n' + source[point:start] + name + source[end:]
        else:
            line, line_start = _line(source, start)
            after = source[:line_start] + f'{indent}{name} = {text}\n' + source[line_start:start] + name + source[end:]
        return FileEdit(path, source, after, 1)
    if kind not in {'function', 'method'}: raise ValueError('unknown extraction')
    if source[_line(source, start)[1]:start].strip() or source[end:source.find('\n', end) if source.find('\n', end) >= 0 else len(source)].strip():
        raise ValueError('functions require complete statement lines')
    # the selection carries its own indentation, and `ast` will not parse an indented block
    block = textwrap.dedent(text)
    try: selected = ast.parse(block).body
    except SyntaxError as e: raise ValueError(f'select whole statements to extract ({e.msg})') from e
    if not selected: raise ValueError('select statements')
    if any(isinstance(n, (ast.Return, ast.Yield, ast.YieldFrom)) for node in selected for n in ast.walk(node)):
        raise ValueError('a selection that returns or yields cannot be lifted into its own function')
    params, stores = _reads_before_writes(selected), _writes(selected)
    params = [p for p in params if p not in _BUILTINS]
    returns = sorted(n for n in stores if n in _reads(source[end:]))
    body = ''.join(indent + '    ' + line if line.strip() else line for line in block.splitlines(True))
    if not body.endswith('\n'): body += '\n'   # a selection can end mid-line; a body cannot
    if returns: body += f'{indent}    return {", ".join(returns)}\n'
    args = list(params)
    if kind != 'method':
        call, head = f'{name}({", ".join(args)})', ', '.join(args)
        if returns: call = f'{", ".join(returns)} = {call}'
        definition = f'{indent}def {name}({head}):\n{body}'
        return FileEdit(path, source, source[:start] + definition + '\n' + indent + call + '\n' + source[end:], 1)
    # a method has to be a sibling of the one it came out of: nested inside it, `self.name` is
    # not an attribute of anything
    args = [p for p in args if p != 'self']
    call = f'self.{name}({", ".join(args)})'
    if returns: call = f'{", ".join(returns)} = {call}'
    holder, member = _enclosing_method(source, start)
    body = textwrap.indent(textwrap.dedent(body), member + '    ')
    definition = f'\n\n{member}def {name}(self{", " if args else ""}{", ".join(args)}):\n{body.rstrip()}\n'
    after = source[:start] + indent + call + '\n' + source[end:]
    at = _offset_after(after, holder)
    return FileEdit(path, source, after[:at] + definition + after[at:], 1)

A lifted function takes what the selection read before it bound, and hands back what the code below it still reads. Everything else stays local.

In [ ]:
F = '''def report(rows):
    total = 0
    for r in rows: total += r
    print(total)
'''
sel = F.index('    total = 0'), F.index('total += r') + len('total += r')
print(extract('r.py', F, *sel, 'function', 'tally').after)

def report(rows):
    def tally(rows):
        total = 0
        for r in rows: total += r
        return total

    total = tally(rows)

    print(total)



In [ ]:
#| hide
_e = extract('r.py', F, *sel, 'function', 'tally')
test_eq(_e.edits, 1)
assert 'def tally(rows):' in _e.after and 'total = tally(rows)' in _e.after
test_fail(lambda: extract('r.py', F, 0, 5, 'function', 'x'), contains='complete statement lines')
test_fail(lambda: extract('r.py', 'def f():\n    return 1\n', 9, 21, 'function', 'x'), contains='returns or yields')
test_fail(lambda: extract('r.py', F, 0, 4, 'function', 'not a name'), contains='valid Python identifier')
test_fail(lambda: extract('r.py', F, 0, 4, 'gadget', 'x'), contains='unknown extraction')
test_fail(lambda: extract('r.py', F, 4, 4, 'function', 'x'), contains='select code before extracting')

A method lands beside the one it came from, not nested inside it, because `self.step` has to be an attribute of the class.

In [ ]:
M = '''class Report:
    def show(self):
        total = self.rows + 1
        print(total)
'''
print(extract('m.py', M, M.index('        total'), M.index('+ 1') + 3, 'method', 'step').after)

class Report:
    def show(self):
        total = self.step()

        print(total)

    def step(self):
        total = self.rows + 1
        return total




In [ ]:
#| hide
V = 'def f(n):\n    return n * (2 + 3)\n'
test_eq(extract('v.py', V, V.index('(2 + 3)'), len(V) - 1, 'variable', 'k').after,
        'def f(n):\n    k = (2 + 3)\n    return n * k\n')
C = '"doc"\nimport os\n\ndef f(p): return open(p, encoding="utf-8")\n'
_sel = C.index('"utf-8"'), len(C) - 2
test_eq(extract('c.py', C, *_sel, 'constant', 'encoding').after,
        '"doc"\nimport os\nENCODING = "utf-8"\n\ndef f(p): return open(p, encoding=ENCODING)\n')
test_eq(extract('c.py', C, *_sel, 'constant', 'ENCODING').after,      # already upper: unchanged
        extract('c.py', C, *_sel, 'constant', 'encoding').after)
test_fail(lambda: extract('c.py', C, *_sel, 'constant', 'naïve'), contains='UPPER_CASE')
test_fail(lambda: extract('c.py', C, C.index('os'), C.index('os') + 2, 'constant', 'x'), contains='literal values')
test_fail(lambda: extract('v.py', V, V.index('n *'), V.index('* (') + 1, 'variable', 'k'), contains='one complete Python expression')
test_fail(lambda: extract('m.py', 'x = 1\ny = 2\n', 0, 5, 'method', 'step'), contains="class's own methods")

## Inline

The reverse: fold a variable's one assignment back into its uses. A value that is not a single atom is parenthesised, so `x = 1 + 2` inlined into `x * 3` gives `(1 + 2) * 3` and not `1 + 2 * 3`.

In [ ]:
#| export
_ATOMIC = (ast.Name, ast.Constant, ast.Attribute, ast.Subscript, ast.Call,
           ast.List, ast.Dict, ast.Set, ast.ListComp, ast.DictComp, ast.SetComp, ast.JoinedStr)

def _binding(tree, name):
    "The one plain `name = value` statement that defines `name`, or a reason there isn't one."
    if any(name in n.names for n in ast.walk(tree) if isinstance(n, (ast.Global, ast.Nonlocal))):
        raise ValueError(f'{name} is declared global or nonlocal')
    stores = [n for n in ast.walk(tree) if isinstance(n, ast.Name) and n.id == name
              and isinstance(n.ctx, (ast.Store, ast.Del))]
    plain = [n for n in ast.walk(tree) if isinstance(n, ast.Assign) and len(n.targets) == 1
             and isinstance(n.targets[0], ast.Name) and n.targets[0].id == name]
    if len(plain) != 1 or len(stores) != 1:
        raise ValueError(f'inline needs {name} assigned exactly once, as a plain `{name} = value`')
    return plain[0]

def inline(path, source, pos):
    "Replace a variable's uses with its value and delete the assignment."
    pos = int(pos)
    if not 0 <= pos <= len(source): raise ValueError('put the cursor on a variable first')
    tree, starts = ast.parse(source), _starts(source)
    at = partial(_at, source, starts)
    spans = {n: at(n) for n in ast.walk(tree) if isinstance(n, ast.Name)}
    here = first(n for n, (a, b) in spans.items() if a <= pos <= b)
    if here is None: raise ValueError('put the cursor on a variable first')
    assign = _binding(tree, here.id)
    astart, aend = at(assign)
    lstart, lend = starts[assign.lineno - 1], starts[assign.end_lineno]
    if source[lstart:astart].strip() or source[aend:lend].strip():
        raise ValueError('inline needs the assignment on lines of its own')
    value = source[slice(*at(assign.value))]
    if not isinstance(assign.value, _ATOMIC): value = f'({value})'
    uses = [span for n, span in spans.items() if n.id == here.id and isinstance(n.ctx, ast.Load)]
    if not uses: raise ValueError(f'{here.id} is never read, so there is nothing to inline')
    if any(a < aend for a, _ in uses): raise ValueError(f'{here.id} is read before it is assigned')
    after = source
    for a, b in sorted(uses, reverse=True): after = after[:a] + value + after[b:]
    return FileEdit(path, source, after[:lstart] + after[lend:], len(uses))

In [ ]:
I = 'width = 1 + 2\nprint(width * 3, width)\n'
inline('i.py', I, I.index('width')).after

'print((1 + 2) * 3, (1 + 2))\n'

In [ ]:
#| hide
test_eq(inline('i.py', I, I.index('width')).edits, 2)
test_eq(inline('i.py', 'w = fn(1)\nprint(w)\n', 0).after, 'print(fn(1))\n')      # a call needs no parentheses
test_fail(lambda: inline('i.py', 'w = 1\nw = 2\nprint(w)\n', 0), contains='assigned exactly once')
test_fail(lambda: inline('i.py', 'global w\nw = 1\nprint(w)\n', 9), contains='global or nonlocal')
test_fail(lambda: inline('i.py', 'w = 1\n', 0), contains='never read')
test_fail(lambda: inline('i.py', 'x = 1\n', 99), contains='put the cursor on a variable first')

## Imports

Where a module can be named relatively, it is: `_stmt` walks the shared package prefix off the front and turns what is left into the right number of dots. `_absolute` goes the other way, resolving an existing relative import against the module that holds it, so `from .a import x` inside `pkg.c` and `from pkg.a import x` inside `pkg.d` are recognised as the same import.

In [ ]:
#| export
def _absolute(node, here):
    "The absolute module an `ImportFrom` names, resolving `level` against the module holding it."
    if not node.level: return node.module or ''
    return '.'.join([*here.split('.')[:-node.level], *([node.module] if node.module else [])])

def _stmt(target, here, names):
    "A `from ... import` reaching `target` from `here`, relative where a shared package allows one."
    tp, hp = target.split('.'), here.split('.')[:-1]
    n = 0
    while n < len(tp) - 1 and n < len(hp) and tp[n] == hp[n]: n += 1
    dots = '.' * (len(hp) - n + 1) if n else ''
    return f"from {dots}{'.'.join(tp[n:])} import {', '.join(names)}"

def _spec(alias): return f'{alias.name} as {alias.asname}' if alias.asname else alias.name
def _from(node, aliases): return f"from {'.' * node.level}{node.module or ''} import {', '.join(_spec(a) for a in aliases)}"

In [ ]:
print(_stmt('pkg.b', 'pkg.c', ['shared']))          # same package: relative
print(_stmt('other.b', 'pkg.c', ['shared']))        # nothing shared: absolute
print(_absolute(ast.parse('from .a import x').body[0], 'pkg.c'))

from .b import shared
from other.b import shared
pkg.a


In [ ]:
#| hide
test_eq(_stmt('pkg.sub.b', 'pkg.c', ['x']), 'from .sub.b import x')
test_eq(_stmt('pkg', 'pkg.c', ['x']), 'from pkg import x')                       # the package itself stays absolute
test_eq(_absolute(ast.parse('from ..a import x').body[0], 'pkg.sub.c'), 'pkg.a')
test_eq(_absolute(ast.parse('from . import x').body[0], 'pkg.c'), 'pkg')
test_eq(_absolute(ast.parse('from os.path import sep').body[0], 'pkg.c'), 'os.path')
test_eq(_from(ast.parse('from .a import x as y, z').body[0], ast.parse('from .a import z').body[0].names),
        'from .a import z')

A new import joins the block the file already has: an existing `from` on the same module grows a name rather than being repeated, and anything genuinely new lands below the docstring, the imports and the `__all__` an nbdev module leads with.

In [ ]:
#| export
def _is_all(n):
    return (isinstance(n, ast.Assign) and len(n.targets) == 1
            and isinstance(n.targets[0], ast.Name) and n.targets[0].id == '__all__')

def _import_point(tree, starts):
    "Where a new import goes: below the docstring, and below the header of imports and `__all__`."
    body, at, i = tree.body, 0, 0
    if body and isinstance(body[0], ast.Expr) and isinstance(getattr(body[0].value, 'value', None), str):
        at, i = starts[body[0].end_lineno], 1
    while i < len(body) and (isinstance(body[i], (ast.Import, ast.ImportFrom)) or _is_all(body[i])):
        at, i = starts[body[i].end_lineno], i + 1
    return at

def _froms(tree, here, module):
    "Every top-level `from module import ...` in the file, with the specs each one carries."
    return [(n, {_spec(a) for a in n.names}) for n in tree.body
            if isinstance(n, ast.ImportFrom) and _absolute(n, here) == module]

def _add_imports(text, tree, starts, here, froms, plains):
    "Edits folding `froms` into whatever import block the file already has, plus the plain imports."
    edits, add = [], []
    plain_texts = {f'import {_spec(a)}' for n in ast.walk(tree) if isinstance(n, ast.Import) for a in n.names}
    for module, specs in sorted(froms.items()):
        if module == here: continue
        block = _froms(tree, here, module)
        node, have = first((n, h) for n, h in block if specs & h) or first(block) or (None, set())
        want = sorted(have | set(specs))
        if want == sorted(have): continue
        if node is not None: edits.append((*_at(text, starts, node), _stmt(module, here, want)))
        else: add.append(_stmt(module, here, sorted(specs)))
    add += [p for p in sorted(plains) if p not in plain_texts]
    if add:
        at = _import_point(tree, starts)
        gap = '' if text[at:at + 1] in ('\n', '') else '\n'   # an import block never runs straight into a definition
        edits.append((at, at, ''.join(s + '\n' for s in add) + gap))
    return edits

def _apply(text, edits):
    for a, b, new in sorted(edits, reverse=True): text = text[:a] + new + text[b:]
    return text

In [ ]:
_t = '"doc"\nimport os\nfrom pkg.b import one\n\nprint(one)\n'
_p = _parse('c.py', _t)
print(_apply(_t, _add_imports(_t, _p, _starts(_t), 'pkg.c', {'pkg.b': {'two'}, 'pkg.d': {'three'}}, {'import sys'})))

"doc"
import os
from .b import one, two
from .d import three
import sys

print(one)



In [ ]:
#| hide
_e = _add_imports(_t, _p, _starts(_t), 'pkg.c', {'pkg.b': {'one'}}, set())
test_eq(_e, [])                                                        # already imported: nothing to do
test_eq(_add_imports(_t, _p, _starts(_t), 'pkg.c', {'pkg.c': {'x'}}, set()), [])   # a file never imports itself
test_eq(_apply('abcdef', [(0, 1, 'X'), (4, 6, 'Y')]), 'XbcdY')

## Moving a definition

`__all__` is rewritten when the file has one this can read: a plain list or tuple of string literals. Anything else is left alone rather than guessed at.

In [ ]:
#| export
def _all_edit(text, starts, tree, add=(), drop=()):
    "An edit rewriting a module's `__all__`, or None when it has none this can read."
    node = first(n for n in tree.body if _is_all(n))
    if node is None or not isinstance(node.value, (ast.List, ast.Tuple)): return None
    cur = [e.value for e in node.value.elts if isinstance(e, ast.Constant) and isinstance(e.value, str)]
    if len(cur) != len(node.value.elts): return None
    new = [x for x in cur if x not in set(drop)] + [x for x in add if x not in cur]
    if new == cur: return None
    return (*_at(text, starts, node.value), '[' + ', '.join(repr(x) for x in new) + ']')

def _carry(froms, plains, node, name, here):
    "Record the import that gave `name` to the source file, so the destination gains it too."
    alias = first(a for a in node.names if (a.asname or a.name).split('.')[0] == name)
    if isinstance(node, ast.ImportFrom): froms.setdefault(_absolute(node, here), set()).add(_spec(alias))
    else: plains.add(f'import {_spec(alias)}')

In [ ]:
#| hide
_a = "__all__ = ['x', 'y']\n"
test_eq(_all_edit(_a, _starts(_a), ast.parse(_a), add=['z'], drop=['x']), (10, 20, "['y', 'z']"))
test_eq(_all_edit(_a, _starts(_a), ast.parse(_a), add=['x']), None)                  # already there
test_eq(_all_edit('__all__ = names\n', [0, 16], ast.parse('__all__ = names\n')), None)
_b = "__all__ = ['x'] + more\n"
test_eq(_all_edit(_b, _starts(_b), ast.parse(_b), add=['z']), None)                  # not a plain list

A call site is repointed two ways. `from src import name` is split, so the names that stayed keep their old import and the moved ones get a new one; `import src` with `src.name` uses is rewritten to a direct import, and the `import src` line is dropped only when nothing else in the file used it. A file that already binds the moved name is left alone with a note, because rewriting it would shadow the file's own definition.

In [ ]:
#| export
def _repoint(path, text, src_mod, dest_mod, names):
    "One file's imports of `names`, pointed at `dest_mod`."
    here, moved = module_of(path), set(names)
    tree, starts = _parse(path, text), _starts(text)
    bound, edits, hit, notes, gained = _bound(tree), [], False, [], set()
    for node in ast.walk(tree):
        if not isinstance(node, ast.ImportFrom) or _absolute(node, here) != src_mod: continue
        taken = [a for a in node.names if a.name in moved]
        if not taken: continue
        hit = True
        keep = [a for a in node.names if a.name not in moved]
        a, b = _at(text, starts, node)
        indent = text[starts[node.lineno - 1]:a]
        lines = ([_from(node, keep)] if keep else []) + [_stmt(dest_mod, here, [_spec(x) for x in taken])]
        edits.append((a, b, ('\n' + indent).join(lines)))
    prefixes = {(a.asname or a.name): (n, a) for n in ast.walk(tree) if isinstance(n, ast.Import)
                for a in n.names if a.name == src_mod}
    for prefix, (node, alias) in prefixes.items():
        root = prefix.split('.')[0]
        reached = [n for n in ast.walk(tree) if isinstance(n, ast.Attribute) and n.attr in moved
                   and ''.join(text[slice(*_at(text, starts, n))].split()) == f'{prefix}.{n.attr}']
        if not reached: continue
        clash = sorted({n.attr for n in reached} & set(bound))
        if clash:
            notes.append(f'{Path(path).name} already binds {", ".join(clash)}; its `{prefix}.` uses were left alone')
            continue
        hit = True
        for n in reached: edits.append((*_at(text, starts, n), n.attr))
        gained |= {n.attr for n in reached}
        loads = sum(1 for n in ast.walk(tree) if isinstance(n, ast.Name) and n.id == root and isinstance(n.ctx, ast.Load))
        if loads == len(reached):
            if len(node.names) == 1: edits.append((starts[node.lineno - 1], starts[node.end_lineno], ''))
            else: edits.append((*_at(text, starts, node),
                                'import ' + ', '.join(_spec(x) for x in node.names if x is not alias)))
    if gained: edits += _add_imports(text, tree, starts, here, {dest_mod: gained}, set())
    if not hit: return None, notes
    return _apply(text, edits), notes

def _gained(path, before, after):
    "Names `after` reads without binding that `before` did not, which is how a move breaks a file."
    btree, atree = _parse(path, before or ''), _parse(path, after)
    keep = _BUILTINS - set(_bound(btree))   # a definition that shadowed a builtin still has to be found
    return sorted(_free([atree], keep) - _free([btree], keep))

In [ ]:
#| hide
with TemporaryDirectory() as d:
    root = _tree(d, {'pkg/__init__.py': '', 'pkg/c.py': 'from pkg.a import helper, shared\nprint(shared(helper(1)))\n'})
    after, notes = _repoint(root/'pkg/c.py', (root/'pkg/c.py').read_text(), 'pkg.a', 'pkg.b', ['shared'])
    test_eq(after, 'from pkg.a import helper\nfrom .b import shared\nprint(shared(helper(1)))\n')
    test_eq(notes, [])
    test_eq(_repoint(root/'pkg/c.py', 'print(1)\n', 'pkg.a', 'pkg.b', ['shared']), (None, []))   # nothing to do
    test_eq(_gained('c.py', 'def f(): pass\nf()\n', 'f()\n'), ['f'])
    test_eq(_gained('c.py', None, 'print(len(x))\n'), ['x'])                     # builtins are already there
    test_eq(_gained('c.py', 'def filter(): pass\nfilter()\n', 'filter()\n'), ['filter'])

`move_plan` is the whole of it: cut the definitions out of `src` with what they own, carry the imports they needed into `dest`, leave a re-export behind unless `shim=False`, repoint every file in `others`, and refuse the move outright if any file it touched would end up reading a name nothing gives it.

In [ ]:
#| export
def move_plan(src, names, dest, others=(), shim=True, read=_read):
    """Move `names` from `src` into `dest` and repoint every file in `others` that imported them.

    `read` returns a file's text, or None when it does not exist. A destination that did not exist
    comes back with `before` None, which is what makes undo delete it again."""
    src, dest, names, notes = str(src), str(dest), list(names), []
    if not names: raise ValueError('choose a function or class to move')
    if Path(src).resolve() == Path(dest).resolve(): raise ValueError('choose a different file to move into')
    if Path(dest).suffix != '.py': raise ValueError('a move lands in a Python file')
    source = read(src)
    tree, starts = _parse(src, source), _starts(source)
    src_mod, dest_mod = module_of(src), module_of(dest)
    top = {n.name: n for n in tree.body if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef))}
    missing = [n for n in names if n not in top]
    if missing: raise ValueError(f"{', '.join(missing)} is not a top-level function or class in {Path(src).name}")
    moved = [top[n] for n in names]
    spans = sorted(_span(source, starts, n) for n in moved)
    block = ''.join(source[a:b] for a, b in spans).strip('\n') + '\n'
    rest = _apply(source, [(a, b, '') for a, b in spans])
    if rest.strip(): rest = rest.rstrip('\n') + '\n'   # what a definition leaves behind should not end in blank lines
    bound, froms, plains, back = _bound(tree), {}, set(), []
    for name in sorted(_free(moved) - set(names)):
        node = bound.get(name)
        if node is None: continue
        if isinstance(node, (ast.Import, ast.ImportFrom)): _carry(froms, plains, node, name, src_mod)
        else: back.append(name)
    if back: froms.setdefault(src_mod, set()).update(back)

    dtext = read(dest)
    fresh = dtext is None
    dtext = dtext or ''
    dtree, dstarts = _parse(dest, dtext), _starts(dtext)
    clash = [n for n in names if n in _bound(dtree)]
    if clash: raise ValueError(f"{Path(dest).name} already defines {', '.join(clash)}")
    dedits = _add_imports(dtext, dtree, dstarts, dest_mod, froms, plains)
    if (e := _all_edit(dtext, dstarts, dtree, add=names)): dedits.append(e)
    dbody = _apply(dtext, dedits)
    dafter = (dbody.rstrip('\n') + '\n\n\n' + block) if dbody.strip() else block

    rtree, rstarts = _parse(src, rest), _starts(rest)
    used = sorted(n for n in names if n in _free([rtree], _BUILTINS - set(names)))
    want = names if shim else used
    # A re-export the source does not otherwise need is dropped rather than refused: keeping it
    # would import the two modules into each other for nothing.
    if want and back:
        if used: raise ValueError(f'{src_mod} and {dest_mod} would import each other over '
                                  f'{", ".join(back)}; move those as well')
        want = []
        notes.append(f're-export left out: {dest_mod} imports {", ".join(back)} back from {src_mod}')
    sedits = _add_imports(rest, rtree, rstarts, src_mod, {dest_mod: set(want)}, set()) if want else []
    if (e := _all_edit(rest, rstarts, rtree, drop=() if shim else names)): sedits.append(e)
    safter = _apply(rest, sedits)

    rows = [FileEdit(src, source, safter, len(names)),
            FileEdit(dest, None if fresh else dtext, dafter, len(names))]
    for path in others:
        if Path(path).resolve() in (Path(src).resolve(), Path(dest).resolve()): continue
        text = read(path)
        if text is None: continue
        try: after, said = _repoint(path, text, src_mod, dest_mod, names)
        except ValueError as e: notes.append(str(e)); continue
        notes += said
        if after is not None and after != text: rows.append(FileEdit(str(path), text, after, 1))
    for row in rows:
        if (broke := _gained(row.path, row.before, row.after)):
            raise ValueError(f"{Path(row.path).name} would lose {', '.join(broke)}; move what it needs as well")
    return rows, notes

### A move, end to end

Three files: `a.py` defines `shared`, `b.py` is where it is going, and `c.py` imports it by name.

In [ ]:
PROJECT = {'pkg/__init__.py': '',
           'pkg/a.py': '''
                "paths"
                import os

                def helper(x): return x + 1

                # the separator, repeated
                def shared(n): return os.sep * n
                ''',
           'pkg/b.py': '"where shared is going"\n',
           'pkg/c.py': 'from pkg.a import helper, shared\n\nprint(shared(2), helper(1))\n'}

def show(rows, notes=()):
    for r in rows: print(f'==== {Path(r.path).name} ' + ('(new) ' if r.before is None else '') + '='*20); print(r.after)
    for n in notes: print('note:', n)

In [ ]:
with TemporaryDirectory() as d:
    root = _tree(d, PROJECT)
    rows, notes = move_plan(root/'pkg/a.py', ['shared'], root/'pkg/b.py', others=[root/'pkg/c.py'])
    show(rows, notes)

==== a.py ====================
"paths"
import os
from .b import shared

def helper(x): return x + 1

==== b.py ====================
"where shared is going"
import os


# the separator, repeated
def shared(n): return os.sep * n

==== c.py ====================
from pkg.a import helper
from .b import shared

print(shared(2), helper(1))



`a.py` keeps a re-export so anything that imported `shared` from it still works; `shim=False` drops it, and then only the files in `others` are repointed.

In [ ]:
with TemporaryDirectory() as d:
    root = _tree(d, PROJECT)
    rows, notes = move_plan(root/'pkg/a.py', ['shared'], root/'pkg/b.py', others=[root/'pkg/c.py'], shim=False)
    print(rows[0].after)

"paths"
import os

def helper(x): return x + 1



In [ ]:
#| hide
with TemporaryDirectory() as d:
    root = _tree(d, PROJECT)
    rows, notes = move_plan(root/'pkg/a.py', ['shared'], root/'pkg/b.py', others=[root/'pkg/c.py'])
    a, b, c = rows
    test_eq(notes, [])
    test_eq([Path(r.path).name for r in rows], ['a.py', 'b.py', 'c.py'])
    test_eq(a.after, '"paths"\nimport os\nfrom .b import shared\n\ndef helper(x): return x + 1\n')
    test_eq(b.after, '"where shared is going"\nimport os\n\n\n# the separator, repeated\ndef shared(n): return os.sep * n\n')
    test_eq(c.after, 'from pkg.a import helper\nfrom .b import shared\n\nprint(shared(2), helper(1))\n')
    test_eq(b.before, '"where shared is going"\n')                 # b.py existed, so undo restores it
    rows, _ = move_plan(root/'pkg/a.py', ['shared'], root/'pkg/new.py')
    test_eq(rows[1].before, None)                                  # new.py did not, so undo deletes it
    test_eq(rows[1].after, 'import os\n\n\n# the separator, repeated\ndef shared(n): return os.sep * n\n')

### The hard cases

`import pkg.a` reaching `pkg.a.shared` is repointed to a direct import, and the `import pkg.a` line survives only because `helper` is still read through it.

In [ ]:
with TemporaryDirectory() as d:
    root = _tree(d, dict(PROJECT, **{'pkg/c.py': 'import pkg.a\n\nprint(pkg.a.shared(2), pkg.a.helper(1))\n',
                                     'pkg/e.py': 'import pkg.a as pa\n\nprint(pa.shared(2))\n',
                                     'pkg/f.py': 'from .a import shared\n\nprint(shared(2))\n'}))
    others = [root/'pkg/c.py', root/'pkg/e.py', root/'pkg/f.py']
    rows, notes = move_plan(root/'pkg/a.py', ['shared'], root/'pkg/b.py', others=others)
    show(rows[2:], notes)

==== c.py ====================
import pkg.a
from .b import shared

print(shared(2), pkg.a.helper(1))

==== e.py ====================
from .b import shared

print(shared(2))

==== f.py ====================
from .b import shared

print(shared(2))



In [ ]:
#| hide
with TemporaryDirectory() as d:
    root = _tree(d, dict(PROJECT, **{'pkg/c.py': 'import pkg.a\n\nprint(pkg.a.shared(2), pkg.a.helper(1))\n',
                                     'pkg/e.py': 'import pkg.a as pa\n\nprint(pa.shared(2))\n',
                                     'pkg/f.py': 'from .a import shared\n\nprint(shared(2))\n',
                                     'pkg/g.py': 'from pkg.a import helper\n\nprint(helper(1))\n',
                                     'pkg/h.py': 'def shared(n): return n\nimport pkg.a\n\nprint(pkg.a.shared(2))\n',
                                     'pkg/bad.py': 'def broken(:\n'}))
    others = [root/f'pkg/{n}.py' for n in ('c', 'e', 'f', 'g', 'h', 'bad', 'gone')]
    rows, notes = move_plan(root/'pkg/a.py', ['shared'], root/'pkg/b.py', others=others)
    got = {Path(r.path).name: r.after for r in rows}
    test_eq(sorted(got), ['a.py', 'b.py', 'c.py', 'e.py', 'f.py'])          # g.py, h.py and bad.py unchanged
    test_eq(got['c.py'], 'import pkg.a\nfrom .b import shared\n\nprint(shared(2), pkg.a.helper(1))\n')
    test_eq(got['e.py'], 'from .b import shared\n\nprint(shared(2))\n')     # nothing else used `pa`, so it went
    test_eq(got['f.py'], 'from .b import shared\n\nprint(shared(2))\n')     # a relative import stays relative
    test_eq(notes, ['h.py already binds shared; its `pkg.a.` uses were left alone',
                    'bad.py does not parse: invalid syntax (line 1)'])

A definition that shadows a builtin is the case a name-based check gets wrong: strip the builtins out and `filter` looks like a name every file already has, so the source would quietly fall back to `builtins.filter` instead of keeping an import of its own.

In [ ]:
SHADOW = {'pkg/__init__.py': '',
          'pkg/a.py': 'def filter(xs): return [x for x in xs if x]\n\ndef use(xs): return len(filter(xs))\n',
          'pkg/b.py': '"where filter is going"\n'}

with TemporaryDirectory() as d:
    root = _tree(d, SHADOW)
    rows, notes = move_plan(root/'pkg/a.py', ['filter'], root/'pkg/b.py', shim=False)
    print(rows[0].after)

from .b import filter

def use(xs): return len(filter(xs))



In [ ]:
#| hide
with TemporaryDirectory() as d:
    root = _tree(d, SHADOW)
    rows, _ = move_plan(root/'pkg/a.py', ['filter'], root/'pkg/b.py', shim=False)
    test_eq(rows[0].after, 'from .b import filter\n\ndef use(xs): return len(filter(xs))\n')

A move that would make the two modules import each other is refused. The moved code reads `BASE` from the file it is leaving, and what stayed behind still calls it, so no ordering of the two imports works and the caller is told to move `BASE` too.

In [ ]:
CIRCULAR = {'pkg/__init__.py': '',
            'pkg/a.py': 'BASE = 3\n\ndef shared(n): return BASE * n\n\ndef helper(x): return shared(x) + 1\n',
            'pkg/b.py': '"b"\n'}

with TemporaryDirectory() as d:
    root = _tree(d, CIRCULAR)
    test_fail(lambda: move_plan(root/'pkg/a.py', ['shared'], root/'pkg/b.py'),
              contains='pkg.a and pkg.b would import each other over BASE')
    print('refused')

refused


When nothing in the source still calls the moved definition there is no cycle to refuse, only a re-export to leave out, and the plan says so.

In [ ]:
with TemporaryDirectory() as d:
    root = _tree(d, dict(CIRCULAR, **{'pkg/a.py': 'BASE = 3\n\ndef shared(n): return BASE * n\n\ndef helper(x): return x + 1\n'}))
    rows, notes = move_plan(root/'pkg/a.py', ['shared'], root/'pkg/b.py')
    show(rows, notes)

==== a.py ====================
BASE = 3

def helper(x): return x + 1

==== b.py ====================
"b"
from .a import BASE


def shared(n): return BASE * n

note: re-export left out: pkg.b imports BASE back from pkg.a


In [ ]:
#| hide
with TemporaryDirectory() as d:
    root = _tree(d, dict(CIRCULAR, **{'pkg/a.py': 'BASE = 3\n\ndef shared(n): return BASE * n\n\ndef helper(x): return x + 1\n'}))
    rows, notes = move_plan(root/'pkg/a.py', ['shared'], root/'pkg/b.py')
    test_eq(notes, ['re-export left out: pkg.b imports BASE back from pkg.a'])
    test_eq(rows[1].after, '"b"\nfrom .a import BASE\n\n\ndef shared(n): return BASE * n\n')

Moving several definitions at once keeps them in source order, and a class comes across with everything it holds. `__all__` follows on both sides.

In [ ]:
MANY = {'pkg/__init__.py': '',
        'pkg/a.py': "__all__ = ['helper', 'shared', 'Thing']\nimport os\n\n"
                    "def helper(x): return x + 1\n\ndef shared(n): return os.sep * n\n\n"
                    "class Thing:\n    def go(self): return shared(1)\n",
        'pkg/b.py': "__all__ = ['already']\n\ndef already(): pass\n"}

with TemporaryDirectory() as d:
    root = _tree(d, MANY)
    rows, notes = move_plan(root/'pkg/a.py', ['shared', 'Thing'], root/'pkg/b.py', shim=False)
    show(rows, notes)

==== a.py ====================
__all__ = ['helper']
import os

def helper(x): return x + 1

==== b.py ====================
__all__ = ['already', 'shared', 'Thing']
import os

def already(): pass


def shared(n): return os.sep * n

class Thing:
    def go(self): return shared(1)



In [ ]:
#| hide
with TemporaryDirectory() as d:
    root = _tree(d, MANY)
    rows, _ = move_plan(root/'pkg/a.py', ['shared', 'Thing'], root/'pkg/b.py', shim=False)
    test_eq(rows[0].after, "__all__ = ['helper']\nimport os\n\ndef helper(x): return x + 1\n")
    test_eq(rows[1].after, "__all__ = ['already', 'shared', 'Thing']\nimport os\n\ndef already(): pass\n\n\n"
                           "def shared(n): return os.sep * n\n\nclass Thing:\n    def go(self): return shared(1)\n")

Every refusal names the file and the thing it could not do.

In [ ]:
#| hide
with TemporaryDirectory() as d:
    root = _tree(d, dict(PROJECT, **{'pkg/b.py': 'def shared(): pass\n'}))
    a, b = root/'pkg/a.py', root/'pkg/b.py'
    test_fail(lambda: move_plan(a, [], b), contains='choose a function or class to move')
    test_fail(lambda: move_plan(a, ['shared'], a), contains='choose a different file to move into')
    test_fail(lambda: move_plan(a, ['shared'], root/'pkg/b.txt'), contains='a move lands in a Python file')
    test_fail(lambda: move_plan(a, ['nope'], b), contains='nope is not a top-level function or class in a.py')
    test_fail(lambda: move_plan(a, ['shared'], b), contains='b.py already defines shared')
    test_fail(lambda: move_plan(root/'pkg/c.py', ['shared'], b), contains='is not a top-level function or class')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()